# chungcat_run — BƯỚC 2: CHƯNG CẤT NHÃN CỦA THẦY VÀO HỌC TRÒ 0,568B

**Đầu vào:** `nhan_thay.json` từ `thay_label_run.ipynb` (upload lên dataset).
**Đầu ra:** model học trò đã tinh chỉnh + số đo dev300 tầng 1.
**Học trò:** `AITeamVN/Vietnamese_Reranker` = **0,568B** — dưới trần 3B. Thầy KHÔNG có mặt ở đây.

### Ba thứ khác hẳn hai lần fine-tune đã thua (−0,50 rồi −6,50)

| | fine-tune cũ | lượt này |
|---|---|---|
| nhãn | **cứng** — gold=1, còn lại=0 | **mềm** — điểm thật của thầy |
| mất mát | cặp rời (BCE) | **KL trên nhóm 8** — học THỨ TỰ |
| tham số đụng vào | **toàn bộ** model | **chỉ LoRA**, gốc đóng băng |

Chỗ thứ ba quan trọng nhất. Nguyên nhân đã ghi của −6,50 là *huấn luyện tiếp đè lên thứ
AITeamVN vốn làm tốt hơn ta*. LoRA **đóng băng toàn bộ trọng số gốc**, chỉ thêm một lớp
hiệu chỉnh bé — nó **không thể** xoá mất thứ model đã biết. Đây là hàng rào cơ chế, không
phải cẩn thận bằng lời.

### Cổng để đi tiếp (đặt TRƯỚC khi chạy)

Notebook tự chấm dev300 tầng 1 cho **cả hai**: học trò gốc và học trò đã chưng cất, **cùng
một đoạn mã, cùng một rổ** — không so với con số chép từ lượt khác.

> **ĐẠT: R@5 học trò ≥ R@5 gốc + 0,0100.** Không đạt → dừng, đừng chạy bước 3 (15h).



In [ ]:
!pip install -q -U peft accelerate



In [ ]:
# ===== Bước 0: cấu hình =====
import os, sys, json, time, glob, random
import numpy as np, torch, torch.nn.functional as F

HOC_TRO = "AITeamVN/Vietnamese_Reranker"
EXC, K  = 900, 8              # PHẢI khớp thay_label_run.ipynb
LORA_R, LORA_A, LR = 16, 32, 1e-4
EPOCH, BS_NHOM, T_NHIET = 3, 4, 1.0     # BS_NHOM=4 nhóm => 32 cặp mỗi bước
NGUONG = 0.0100               # cổng: hơn gốc bao nhiêu R@5 thì mới đi tiếp
SEED = 20260825

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "CẦN GPU"
INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
CTX_DIR = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",
                           f"{INPUT_DIR}/selected-contexts")
               if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)
def tim(ten):
    """Tìm file trong dataset dù nó nằm ở gốc hay trong thư mục con.
    BẪY 25/08: hardcode "Input/train.json" theo cây thư mục máy cá nhân, nhưng trên Kaggle
    file được upload ra GỐC dataset -> FileNotFoundError sau 22 giây. Đừng đoán cây thư mục."""
    p = glob.glob(f"{INPUT_DIR}/**/{ten}", recursive=True)
    assert p, f"KHÔNG THẤY {ten} trong dataset — upload rồi chạy lại"
    return sorted(p, key=len)[0]

import deep_chunk as DC
from rerank_from_d import blend_bm25_first
DC.MERGE_CHARS = 1800
print("thiết bị:", torch.cuda.get_device_name(0))



In [ ]:
# ===== Bước 1: nạp nhãn thầy, dựng nhóm huấn luyện =====
nf = glob.glob(f"{INPUT_DIR}/**/nhan_thay.json", recursive=True)
assert nf, "CHƯA UPLOAD nhan_thay.json lên dataset"
nhan = json.load(open(nf[0], encoding="utf-8"))
train = json.load(open(tim("train.json"), encoding="utf-8"))
bm25  = json.load(open(tim("bm25_ids_train_FALLBACK.json"), encoding="utf-8"))

du = {q: e for q, e in nhan.items() if len(e) == K}
print(f"{len(nhan)} câu có nhãn · {len(du)} câu ĐỦ {K} nhãn")
assert len(du) >= 300, f"chỉ {len(du)} nhóm đủ — quá ít, chạy nốt bước 1 đã"

# gold PHẢI nằm trong nhóm, nếu không nhóm đó vô nghĩa
nhomtr = []
for q, e in du.items():
    gold = {str(a) for a in train[q]["answer"]}
    if not (gold & set(e)): continue
    nhomtr.append((q, list(e)))
print(f"{len(nhomtr)} nhóm dùng được")

t0 = time.time(); X, Y = [], []
for i, (q, docs) in enumerate(nhomtr, 1):
    qs = train[q]["question"]
    txt = [(DC.pick_chunks(qs, CTX_DIR, d, k=1) or [""])[0][:EXC] for d in docs]
    p = np.clip(np.array([nhan[q][d] for d in docs], dtype=np.float64), 1e-6, 1 - 1e-6)
    # điểm thầy là XÁC SUẤT P("yes"). Đưa về logit rồi mới softmax: giữ nguyên thứ tự mà
    # trải đúng khoảng cách. Softmax thẳng trên xác suất thì mọi nhóm gần như phẳng.
    X.append([[qs, t] for t in txt]); Y.append(np.log(p / (1 - p)))
    if i % 200 == 0: print(f"  băm {i}/{len(nhomtr)} | {(time.time()-t0)/60:.1f} phút", flush=True)
Y = torch.tensor(np.array(Y), dtype=torch.float32)
print(f"\nxong {len(X)} nhóm · {(time.time()-t0)/60:.1f} phút")



In [ ]:
# ===== Bước 2: bộ chấm dev300 — DÙNG CHUNG cho gốc và bản chưng cất =====
dev  = json.load(open(tim("dev_300_locked.json"), encoding="utf-8"))
cand = json.load(open(tim("fusion_rrf_top50_dev_1000_matchEmbedded.json"), encoding="utf-8"))
qd   = [q for q in dev if q in cand]
GOLD = {q: {str(a) for a in dev[q]["answer"]} for q in qd}
ORDER= {q: [str(c["doc_id"]) for c in sorted(cand[q], key=lambda c: -float(c["rrf_score"]))]
        for q in qd}
t0 = time.time(); dX, dI = [], []
for i, q in enumerate(qd, 1):
    qs = dev[q]["question"]
    for d in ORDER[q]:
        dX.append([qs, (DC.pick_chunks(qs, CTX_DIR, d, k=1) or [""])[0][:EXC]]); dI.append((q, d))
    if i % 100 == 0: print(f"  băm dev {i}/{len(qd)} | {(time.time()-t0)/60:.1f} phút", flush=True)
print(f"{len(dX):,} cặp dev\n")

@torch.no_grad()
def cham_dev(model, tok, bs=64):
    model.eval(); out = []
    for i in range(0, len(dX), bs):
        b = dX[i:i+bs]
        enc = tok([x[0] for x in b], [x[1] for x in b], padding=True, truncation=True,
                  max_length=512, return_tensors="pt").to("cuda")
        out += model(**enc).logits.squeeze(-1).float().cpu().tolist()
    per = {}
    for (q, d), v in zip(dI, out): per.setdefault(q, {})[d] = v
    r = {1: 0.0, 5: 0.0}
    for q in qd:
        p = blend_bm25_first(sorted(per[q], key=lambda d: -per[q][d]), ORDER[q], k=5, n_bm25=1)
        for k in r: r[k] += len(GOLD[q] & set(p[:k])) / len(GOLD[q])
    return {k: v / len(qd) for k, v in r.items()}, per



In [ ]:
# ===== Bước 3: đo học trò GỐC (mốc so sánh, cùng đoạn mã) =====
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok  = AutoTokenizer.from_pretrained(HOC_TRO)
base = AutoModelForSequenceClassification.from_pretrained(HOC_TRO).to("cuda")
n = sum(p.numel() for p in base.parameters())
print(f"học trò: {n:,} ({n/1e9:.3f}B)")
assert n <= 3_000_000_000, "học trò vượt trần 3B"

t0 = time.time(); MOC, _ = cham_dev(base, tok)
print(f"\nGỐC   R@1={MOC[1]:.4f}  R@5={MOC[5]:.4f}   ({(time.time()-t0)/60:.1f} phút)")



In [ ]:
# ===== Bước 4: chưng cất bằng LoRA =====
from peft import LoraConfig, get_peft_model
try:
    cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_A, lora_dropout=0.05, bias="none",
                     task_type="SEQ_CLS", target_modules="all-linear")
    hoc = get_peft_model(base, cfg)
except Exception as e:
    print(f"(all-linear không dùng được: {type(e).__name__}) -> chỉ định tay cho XLM-R")
    cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_A, lora_dropout=0.05, bias="none",
                     task_type="SEQ_CLS", target_modules=["query", "key", "value", "dense"])
    hoc = get_peft_model(base, cfg)
hoc.print_trainable_parameters()

opt = torch.optim.AdamW([p for p in hoc.parameters() if p.requires_grad], lr=LR)
idx = list(range(len(X)))
buoc = EPOCH * ((len(idx) + BS_NHOM - 1) // BS_NHOM)
sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=buoc, pct_start=0.1)
scaler = torch.amp.GradScaler("cuda")
print(f"{buoc} bước · {EPOCH} epoch · {BS_NHOM} nhóm/bước\n")

t0 = time.time(); b = 0
for ep in range(EPOCH):
    random.shuffle(idx); tong = 0.0
    hoc.train()
    for j in range(0, len(idx), BS_NHOM):
        lo = idx[j:j+BS_NHOM]
        pairs = [p for i in lo for p in X[i]]
        enc = tok([p[0] for p in pairs], [p[1] for p in pairs], padding=True,
                  truncation=True, max_length=512, return_tensors="pt").to("cuda")
        with torch.amp.autocast("cuda", dtype=torch.float16):
            lg = hoc(**enc).logits.squeeze(-1).view(len(lo), K)
        # KL(thầy ‖ trò) trên softmax của NHÓM. Học THỨ TỰ, không học điểm tuyệt đối.
        tgt = F.softmax(Y[lo].to("cuda") / T_NHIET, dim=-1)
        loss = F.kl_div(F.log_softmax(lg.float(), dim=-1), tgt, reduction="batchmean")
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
        tong += loss.item(); b += 1
        if b % 50 == 0: print(f"  bước {b}/{buoc} · loss {tong/(j//BS_NHOM+1):.4f} · "
                              f"{(time.time()-t0)/60:.1f} phút", flush=True)
    print(f"epoch {ep+1}: loss {tong/((len(idx)+BS_NHOM-1)//BS_NHOM):.4f}")
print(f"\nhuấn luyện xong · {(time.time()-t0)/60:.1f} phút")



In [ ]:
# ===== Bước 5: đo lại, so với mốc, in cổng =====
MOI, per = cham_dev(hoc, tok)
d1, d5 = MOI[1] - MOC[1], MOI[5] - MOC[5]
print(f"\n{'':10}{'R@1':>9}{'R@5':>9}")
print(f"{'gốc':<10}{MOC[1]:>9.4f}{MOC[5]:>9.4f}")
print(f"{'chưng cất':<10}{MOI[1]:>9.4f}{MOI[5]:>9.4f}")
print(f"{'Δ':<10}{d1:>+9.4f}{d5:>+9.4f}")
dat = d5 >= NGUONG
ket = ("✅ ĐẠT — chạy bước 3, mở rộng 2.000-3.000 câu" if dat else
       "❌ KHÔNG ĐẠT — DỪNG, đừng đốt 15h cho bước 3. Báo lại số này.")
print(f"\nCỔNG (R@5 >= gốc + {NGUONG:.4f}): {ket}")

json.dump(per, open(f"{OUT}/scores_dev300_chungcat_tang1.json", "w", encoding="utf-8"),
          ensure_ascii=False)
json.dump({"moc": MOC, "moi": MOI, "delta_r5": d5, "dat_cong": bool(dat),
           "n_nhom": len(X), "lora_r": LORA_R, "lr": LR, "epoch": EPOCH, "T": T_NHIET},
          open(f"{OUT}/meta_chungcat.json", "w", encoding="utf-8"))
if dat:
    hoc.save_pretrained(f"{OUT}/lora_chungcat"); tok.save_pretrained(f"{OUT}/lora_chungcat")
    print(f"đã lưu adapter -> {OUT}/lora_chungcat (vài chục MB, tải về được)")
else:
    print("KHÔNG lưu adapter — không đạt cổng thì không có gì đáng giữ.")
print("\nTẢI outputs/ VỀ TRƯỚC KHI ĐÓNG PHIÊN.")


